In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.medallion_data.ml_cache;

In [0]:
import os

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/medallion_data/ml_cache"


In [0]:
# ============================================================
# 0. IMPORTS
# MAGIC GAMMA TELESCOPE
# PySpark ML MLOps Notebook - Logistic Regression
# ============================================================

import mlflow
import mlflow.pyspark.ml

from pyspark.ml.connect.classification import LogisticRegression
from pyspark.ml.connect.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.sql import functions as F


In [0]:
# ============================================================
# 1. LOAD GOLD TABLE
# ============================================================

gold_table_name = "workspace.medallion_data.gold_telescope"

gold_df = spark.table(gold_table_name)

display(gold_df.limit(10))
gold_df.printSchema()

print("Gold table loaded:", gold_table_name)
print("Gold row count:", gold_df.count())
print("Gold column count:", len(gold_df.columns))

In [0]:
# ============================================================
# 2. CHECK LABEL DISTRIBUTION
# ============================================================

display(
    gold_df
    .groupBy("label")
    .count()
    .orderBy("label")
)

In [0]:
# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = gold_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

In [0]:
# ============================================================
# 4. DEFINE PYSPARK LOGISTIC REGRESSION MODEL
# ============================================================

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100
)

In [0]:
# ============================================================
# 5. TRAIN LOGISTIC REGRESSION WITH MLFLOW
# Serverless-safe version: NO autolog, NO model registration yet
# ============================================================

import mlflow
from pyspark.ml.classification import LogisticRegression as ClassicLogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator as ClassicMulticlassClassificationEvaluator, BinaryClassificationEvaluator as ClassicBinaryClassificationEvaluator

mlflow.set_experiment("/Users/maria.laramoran27@ncf.edu/gamma_telescope_pyspark_mlops")

with mlflow.start_run(run_name="Telescope_Production_Model_PySpark_LogisticRegression") as run:

    # Train model
    lr = ClassicLogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        maxIter=100
    )
    model = lr.fit(train_df)

    # Generate predictions
    predictions = model.transform(test_df)

    # Evaluators
    accuracy_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )

    f1_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1"
    )

    precision_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    recall_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall"
    )

    auc_evaluator = ClassicBinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    # Metrics
    test_accuracy = accuracy_evaluator.evaluate(predictions)
    test_f1 = f1_evaluator.evaluate(predictions)
    test_precision = precision_evaluator.evaluate(predictions)
    test_recall = recall_evaluator.evaluate(predictions)
    test_auc = auc_evaluator.evaluate(predictions)

    # Log params manually
    mlflow.log_param("model_type", "PySpark LogisticRegression")
    mlflow.log_param("gold_table", gold_table_name)
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("regParam", 0.01)
    mlflow.log_param("elasticNetParam", 0.0)

    # Log metrics manually
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_precision_weighted", test_precision)
    mlflow.log_metric("test_recall_weighted", test_recall)
    mlflow.log_metric("test_f1_weighted", test_f1)
    mlflow.log_metric("test_auc", test_auc)

    print("Test Metrics:")
    print(f"test_accuracy: {test_accuracy:.4f}")
    print(f"test_precision_weighted: {test_precision:.4f}")
    print(f"test_recall_weighted: {test_recall:.4f}")
    print(f"test_f1_weighted: {test_f1:.4f}")
    print(f"test_auc: {test_auc:.4f}")

    run_id = run.info.run_id
    print("MLflow Run ID:", run_id)

In [0]:
# ============================================================
# 6. TRAIN & TUNE LOGISTIC REGRESSION WITH CROSS-VALIDATION & MLFLOW
# Serverless-safe version: NO autolog, NO model registration yet
# ============================================================

import mlflow
from pyspark.ml.classification import LogisticRegression as ClassicLogisticRegression
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator as ClassicMulticlassClassificationEvaluator,
    BinaryClassificationEvaluator as ClassicBinaryClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

mlflow.set_experiment("/Users/maria.laramoran27@ncf.edu/gamma_telescope_pyspark_mlops")

with mlflow.start_run(run_name="Telescope_Production_Model_PySpark_LR_CV") as run:

    # 1. Define the base estimator
    lr = ClassicLogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        maxIter=100
    )

    # 2. Build the parameter grid for hyperparameter tuning
    paramGrid = (
        ParamGridBuilder()
        .addGrid(lr.regParam, [0.01, 0.1, 1.0])
        .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
        .build()
    )

    # 3. Define the evaluator for Cross Validation
    cv_evaluator = ClassicBinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    # 4. Set up the CrossValidator
    cv = CrossValidator(
        estimator=lr,
        estimatorParamMaps=paramGrid,
        evaluator=cv_evaluator,
        numFolds=5,
        parallelism=2,
        seed=42
    )

    # 5. Fit CrossValidator to the training data
    print("Starting 5-fold Cross-Validation... this may take a moment.")
    cv_model = cv.fit(train_df)
    best_model = cv_model.bestModel

    # 6. Generate predictions for both training and holdout test sets
    train_predictions = best_model.transform(train_df)
    test_predictions = best_model.transform(test_df)

    # 7. Define evaluators for final metrics
    accuracy_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    f1_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1"
    )
    precision_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
    )
    recall_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedRecall"
    )
    auc_evaluator = ClassicBinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )

    # 8. Calculate comparable training and test metrics
    train_accuracy = accuracy_evaluator.evaluate(train_predictions)
    train_f1 = f1_evaluator.evaluate(train_predictions)
    train_precision = precision_evaluator.evaluate(train_predictions)
    train_recall = recall_evaluator.evaluate(train_predictions)
    train_auc = auc_evaluator.evaluate(train_predictions)

    test_accuracy = accuracy_evaluator.evaluate(test_predictions)
    test_f1 = f1_evaluator.evaluate(test_predictions)
    test_precision = precision_evaluator.evaluate(test_predictions)
    test_recall = recall_evaluator.evaluate(test_predictions)
    test_auc = auc_evaluator.evaluate(test_predictions)

    # 9. Extract best hyperparameters and cross-validation score
    best_reg_param = best_model.getRegParam()
    best_elastic_net_param = best_model.getElasticNetParam()
    best_max_iter = best_model.getMaxIter()
    best_cv_auc = max(cv_model.avgMetrics)

    # 10. Log params manually to MLflow
    mlflow.log_param("model_type", "PySpark LogisticRegression CV")
    mlflow.log_param("gold_table", gold_table_name)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("maxIter", best_max_iter)
    mlflow.log_param("best_regParam", best_reg_param)
    mlflow.log_param("best_elasticNetParam", best_elastic_net_param)

    # 11. Log metrics manually to MLflow
    mlflow.log_metric("cv_best_avg_auc", best_cv_auc)
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_precision_weighted", train_precision)
    mlflow.log_metric("train_recall_weighted", train_recall)
    mlflow.log_metric("train_f1_weighted", train_f1)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_precision_weighted", test_precision)
    mlflow.log_metric("test_recall_weighted", test_recall)
    mlflow.log_metric("test_f1_weighted", test_f1)
    mlflow.log_metric("test_auc", test_auc)

    print("\n--- Best Model Parameters ---")
    print(f"regParam: {best_reg_param}")
    print(f"elasticNetParam: {best_elastic_net_param}")
    print(f"best_cv_avg_auc: {best_cv_auc:.4f}")

    print("\n--- Training Metrics ---")
    print(f"train_accuracy: {train_accuracy:.4f}")
    print(f"train_precision_weighted: {train_precision:.4f}")
    print(f"train_recall_weighted: {train_recall:.4f}")
    print(f"train_f1_weighted: {train_f1:.4f}")
    print(f"train_auc: {train_auc:.4f}")

    print("\n--- Test Metrics ---")
    print(f"test_accuracy: {test_accuracy:.4f}")
    print(f"test_precision_weighted: {test_precision:.4f}")
    print(f"test_recall_weighted: {test_recall:.4f}")
    print(f"test_f1_weighted: {test_f1:.4f}")
    print(f"test_auc: {test_auc:.4f}")

    run_id = run.info.run_id
    print("\nMLflow Run ID:", run_id)

    # 12. Serverless/shared compute needs a UC Volume temp path
    import os

    catalog_name = "workspace"
    schema_name = "medallion_data"
    volume_name = "mlflow_tmp"

    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

    mlflow_dfs_tmp = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/sparkml_tmp"
    os.environ["MLFLOW_DFS_TMP"] = mlflow_dfs_tmp

    print("\nMLFLOW_DFS_TMP:", mlflow_dfs_tmp)

    # 13. Log the best Spark ML model to MLflow
    # Turn off the complex vector outputs to prevent MLflow serialization bugs
    best_model.setProbabilityCol("")
    best_model.setRawPredictionCol("")

    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="model",
        dfs_tmpdir=mlflow_dfs_tmp
    )

    model_uri = f"runs:/{run_id}/model"

    print("\nModel logged to MLflow.")
    print("Model URI:", model_uri)

In [0]:
# ============================================================
# 7. LOG MODEL WITH SIGNATURE AND REGISTER IN UNITY CATALOG
# Fix for: "Model passed for registration did not contain any signature metadata"
# ============================================================

import os
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_registry_uri("databricks-uc")

# ------------------------------------------------------------
# 1. UC Volume temp path required for Spark ML models
# ------------------------------------------------------------

catalog_name = "workspace"
schema_name = "medallion_data"
volume_name = "mlflow_tmp"

# Create volume if allowed
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

mlflow_dfs_tmp = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/sparkml_tmp"
os.environ["MLFLOW_DFS_TMP"] = mlflow_dfs_tmp

print("MLFLOW_DFS_TMP:", mlflow_dfs_tmp)

# ------------------------------------------------------------
# 2. Create input and output examples for signature
# ------------------------------------------------------------

# Your model expects a Spark ML vector column called "features"
input_sample = test_df.select("features").limit(10)

# Get model outputs
output_sample = (
    best_model
    .transform(input_sample)
    .select("prediction", "probability")
)

# Infer model signature with BOTH input and output schema
signature = infer_signature(input_sample, output_sample)

print("Model signature created:")
print(signature)

# ------------------------------------------------------------
# 3. Log the model again, now WITH signature
# ------------------------------------------------------------

with mlflow.start_run(run_name="Telescope_Production_Model_With_Signature") as signature_run:

    mlflow.log_param("model_type", "PySpark LogisticRegression CV")
    mlflow.log_param("registered_candidate", "true")
    mlflow.log_param("gold_table", gold_table_name)

    # Optional: log your final test metrics again if variables still exist
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_precision_weighted", test_precision)
    mlflow.log_metric("test_recall_weighted", test_recall)
    mlflow.log_metric("test_f1_weighted", test_f1)
    mlflow.log_metric("test_auc", test_auc)

    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="model",
        signature=signature,
        dfs_tmpdir=mlflow_dfs_tmp
    )

    signed_run_id = signature_run.info.run_id
    model_uri = f"runs:/{signed_run_id}/model"

    print("\n--- MODEL LOGGED WITH SIGNATURE ---")
    print("Signed Run ID:", signed_run_id)
    print("Model URI:", model_uri)

# ------------------------------------------------------------
# 4. Register the signed model in Unity Catalog
# ------------------------------------------------------------

registered_model_name = f"{catalog_name}.{schema_name}.gamma_telescope_pyspark_lr"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("\n--- MODEL REGISTERED SUCCESSFULLY ---")
print(f"Registered model name: {registered_model_name}")
print(f"Registered model version: {registered_model.version}")
print(f"Model URI: {model_uri}")

In [0]:
# ============================================================
# 8. DEPLOY REGISTERED MODEL TO DATABRICKS MODEL SERVING
# ============================================================

import mlflow
from mlflow.deployments import get_deploy_client

# Use Unity Catalog Model Registry
mlflow.set_registry_uri("databricks-uc")

# ------------------------------------------------------------
# 1. Define registered model information
# ------------------------------------------------------------

catalog_name = "workspace"
schema_name = "medallion_data"
model_name = "gamma_telescope_pyspark_lr"

registered_model_name = f"{catalog_name}.{schema_name}.{model_name}"


# If registered_model.version still exists, use it.
model_version = str(registered_model.version)

endpoint_name = "gamma-telescope-lr-endpoint"

print("Registered model:", registered_model_name)
print("Model version:", model_version)
print("Endpoint name:", endpoint_name)

# ------------------------------------------------------------
# 2. Create or Update Databricks serving endpoint
# ------------------------------------------------------------

client = get_deploy_client("databricks")

endpoint_config = {
    "served_entities": [
        {
            "name": f"{model_name}-v{model_version}",
            "entity_name": registered_model_name,
            "entity_version": model_version,
            "workload_size": "Small",
            "scale_to_zero_enabled": True
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name": f"{model_name}-v{model_version}",
                "traffic_percentage": 100
            }
        ]
    }
}

try:
    # First, try to get the endpoint to see if it already exists
    client.get_endpoint(endpoint=endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists. Updating with new version...")
    
    # If it exists, use update_endpoint instead of create_endpoint
    endpoint = client.update_endpoint(
        endpoint=endpoint_name,
        config=endpoint_config
    )
    print("\n--- SERVING ENDPOINT UPDATED ---")

except Exception as e:
    # If it throws an error, we assume the endpoint does not exist yet
    print(f"Endpoint '{endpoint_name}' not found. Creating a new one...")
    
    # Use create_endpoint for the initial deployment
    endpoint = client.create_endpoint(
        name=endpoint_name,
        config=endpoint_config
    )
    print("\n--- SERVING ENDPOINT CREATED ---")

print("Endpoint name:", endpoint_name)
print("Serving model:", registered_model_name)
print("Version:", model_version)
print("Status: Provisioning / Updating...")

In [0]:
# ============================================================
# 9. VERIFY DEPLOYMENT ENDPOINT STATUS
# Check that the Databricks Serving Endpoint exists and is ready
# before connecting the external Streamlit app
# ============================================================

from mlflow.deployments import get_deploy_client
import json

# ------------------------------------------------------------
# 1. Define endpoint name
# ------------------------------------------------------------

endpoint_name = "gamma-telescope-lr-endpoint"  # use your exact endpoint name

# ------------------------------------------------------------
# 2. Connect to Databricks Model Serving
# ------------------------------------------------------------

client = get_deploy_client("databricks")

# ------------------------------------------------------------
# 3. Get endpoint information
# ------------------------------------------------------------

try:
    endpoint_info = client.get_endpoint(endpoint=endpoint_name)

    print("\n--- ENDPOINT FOUND ---")
    print(f"Endpoint name: {endpoint_info.get('name', endpoint_name)}")

    # Endpoint state info
    state = endpoint_info.get("state", {})

    ready_state = state.get("ready", "UNKNOWN")
    config_update_state = state.get("config_update", "UNKNOWN")

    print("\n--- ENDPOINT STATUS ---")
    print(f"Ready state: {ready_state}")
    print(f"Config update state: {config_update_state}")

    # --------------------------------------------------------
    # 4. Check if endpoint is ready
    # --------------------------------------------------------

    if ready_state == "READY" and config_update_state == "NOT_UPDATING":
        print("\n Endpoint is READY and can receive prediction requests.")
        print("Next step: connect the Streamlit app to this endpoint.")
    else:
        print("\n Endpoint exists, but it is not fully ready yet.")
        print("Wait until Ready state = READY and Config update state = NOT_UPDATING.")

    # --------------------------------------------------------
    # 5. Print served model information
    # --------------------------------------------------------

    config = endpoint_info.get("config", {})
    served_entities = config.get("served_entities", [])

    print("\n--- SERVED MODEL DETAILS ---")

    if served_entities:
        for entity in served_entities:
            print(f"Served entity name: {entity.get('name')}")
            print(f"Model name: {entity.get('entity_name')}")
            print(f"Model version: {entity.get('entity_version')}")
            print(f"Workload size: {entity.get('workload_size')}")
            print(f"Scale to zero: {entity.get('scale_to_zero_enabled')}")
            print("-" * 50)
    else:
        print("No served entities found in endpoint config.")

    # --------------------------------------------------------
    # 6. Optional: show full endpoint JSON
    # --------------------------------------------------------

    print("\n--- FULL ENDPOINT INFO JSON ---")
    print(json.dumps(endpoint_info, indent=2))

except Exception as e:
    print("\n Endpoint check failed.")
    print("Possible reasons:")
    print("- The endpoint name is misspelled.")
    print("- The endpoint was not created successfully.")
    print("- You do not have permission to view this endpoint.")
    print("- The endpoint is still being created.")
    print("\nError message:")
    print(e)